# Setup & Validation Notebook

**Purpose**: Validate Azure service connections and environment setup

**Task**: T010 - Setup validation per quickstart.md Section 3

**Learning Objectives**:
- Test Azure OpenAI connection (GPT-4o-mini)
- Test Document Intelligence connection
- Test Azure AI Search connection
- Test embeddings model (text-embedding-ada-002)
- Verify environment configuration

**Expected Time**: 10 minutes

---

## 1. Load Configuration

Load environment variables and verify all required credentials are present.

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path for imports
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load configuration
from config import Config

config = Config()

print("✅ Configuration loaded successfully")
print(f"\nAzure OpenAI Endpoint: {config.AZURE_OPENAI_ENDPOINT}")
print(f"GPT-4 Deployment: {config.AZURE_OPENAI_DEPLOYMENT_GPT4}")
print(f"Embeddings Deployment: {config.AZURE_OPENAI_DEPLOYMENT_EMBEDDINGS}")
print(f"Document Intelligence Endpoint: {config.AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT}")
print(f"AI Search Endpoint: {config.AZURE_SEARCH_ENDPOINT}")
print(f"AI Search Index: {config.AZURE_SEARCH_INDEX_NAME}")

## 2. Test Azure OpenAI Connection

Test connection to Azure OpenAI and verify GPT-4o-mini deployment works.

In [ ]:
from openai import AzureOpenAI
import time

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=config.AZURE_OPENAI_API_KEY,
    api_version=config.AZURE_OPENAI_API_VERSION,
    azure_endpoint=config.AZURE_OPENAI_ENDPOINT
)

print("Testing Azure OpenAI connection...\n")

try:
    start_time = time.time()
    
    # Test GPT-4o-mini deployment
    response = client.chat.completions.create(
        model=config.AZURE_OPENAI_DEPLOYMENT_GPT4,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say 'Hello from Azure OpenAI!' and nothing else."}
        ],
        max_tokens=20,
        temperature=0
    )
    
    elapsed = time.time() - start_time
    
    print(f"✅ Azure OpenAI Connection Successful!")
    print(f"\nModel: {response.model}")
    print(f"Response: {response.choices[0].message.content}")
    print(f"Tokens Used: {response.usage.total_tokens}")
    print(f"Response Time: {elapsed:.2f}s")
    
except Exception as e:
    print(f"❌ Azure OpenAI Connection Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_OPENAI_API_KEY in .env file")
    print("2. Verify AZURE_OPENAI_ENDPOINT format (should end with /)")
    print("3. Confirm deployment name matches AZURE_OPENAI_DEPLOYMENT_GPT4")
    print("4. Check Azure Portal → OpenAI resource → Model deployments")

## 3. Test Embeddings Model

Test text-embedding-ada-002 deployment for RAG system.

In [ ]:
print("Testing embeddings model...\n")

try:
    start_time = time.time()
    
    # Test embeddings
    response = client.embeddings.create(
        model=config.AZURE_OPENAI_DEPLOYMENT_EMBEDDINGS,
        input="This is a test document for embedding generation."
    )
    
    elapsed = time.time() - start_time
    embedding = response.data[0].embedding
    
    print(f"✅ Embeddings Model Working!")
    print(f"\nModel: {response.model}")
    print(f"Embedding Dimensions: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
    print(f"Tokens Used: {response.usage.total_tokens}")
    print(f"Response Time: {elapsed:.2f}s")
    
except Exception as e:
    print(f"❌ Embeddings Model Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_OPENAI_DEPLOYMENT_EMBEDDING in .env")
    print("2. Verify text-embedding-ada-002 is deployed in Azure Portal")
    print("3. Ensure deployment name matches config")

## 4. Test Azure Document Intelligence

Test connection to Azure Document Intelligence service.

In [ ]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

print("Testing Azure Document Intelligence...\n")

try:
    # Initialize Document Intelligence client
    doc_client = DocumentAnalysisClient(
        endpoint=config.AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT,
        credential=AzureKeyCredential(config.AZURE_DOCUMENT_INTELLIGENCE_KEY)
    )
    
    # Test with sample URL (Microsoft's sample invoice)
    sample_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"
    
    print("Analyzing sample invoice...")
    start_time = time.time()
    
    poller = doc_client.begin_analyze_document_from_url(
        "prebuilt-invoice",
        sample_url
    )
    result = poller.result()
    elapsed = time.time() - start_time
    
    print(f"✅ Document Intelligence Working!")
    print(f"\nAnalysis completed in {elapsed:.2f}s")
    print(f"Documents analyzed: {len(result.documents)}")
    
    if result.documents:
        doc = result.documents[0]
        print(f"Document type: {doc.doc_type}")
        print(f"Confidence: {doc.confidence:.2%}")
        print(f"Fields extracted: {len(doc.fields)}")
        
        # Show sample fields
        print("\nSample extracted fields:")
        for field_name in list(doc.fields.keys())[:5]:
            field = doc.fields[field_name]
            print(f"  - {field_name}: {field.value if field.value else 'N/A'} (confidence: {field.confidence:.2%})")
    
except Exception as e:
    print(f"❌ Document Intelligence Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_DOCUMENT_INTELLIGENCE_KEY in .env")
    print("2. Verify AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT format")
    print("3. Ensure Document Intelligence resource is provisioned")
    print("4. Check Azure Portal → Document Intelligence → Keys and Endpoint")

## 5. Test Azure AI Search

Test connection to Azure AI Search service.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

print("Testing Azure AI Search...\n")

try:
    # Initialize Search Index client
    search_client = SearchIndexClient(
        endpoint=config.AZURE_SEARCH_ENDPOINT,
        credential=AzureKeyCredential(config.AZURE_SEARCH_ADMIN_KEY)
    )
    
    # List existing indexes
    indexes = list(search_client.list_indexes())
    
    print(f"✅ Azure AI Search Connection Successful!")
    print(f"\nService endpoint: {config.AZURE_SEARCH_ENDPOINT}")
    print(f"Existing indexes: {len(indexes)}")
    
    if indexes:
        print("\nIndex names:")
        for idx in indexes:
            print(f"  - {idx.name}")
    else:
        print("\nNo indexes found (will be created during RAG setup)")
    
    # Check if our target index exists
    target_index = config.AZURE_SEARCH_INDEX_NAME
    index_exists = any(idx.name == target_index for idx in indexes)
    
    if index_exists:
        print(f"\n✅ Target index '{target_index}' already exists")
    else:
        print(f"\nℹ️  Target index '{target_index}' not found (will be created in notebook 03)")
    
except Exception as e:
    print(f"❌ Azure AI Search Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_SEARCH_ADMIN_KEY in .env")
    print("2. Verify AZURE_SEARCH_ENDPOINT format")
    print("3. Ensure AI Search service is provisioned (Free tier is fine)")
    print("4. Check Azure Portal → AI Search → Keys")

## 6. Environment Summary

Display complete setup validation results.

In [ ]:
import json
from datetime import datetime

print("="*70)
print("SETUP VALIDATION SUMMARY")
print("="*70)
print(f"\nValidation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")

print("\n" + "-"*70)
print("AZURE SERVICES STATUS")
print("-"*70)

services = {
    "Azure OpenAI (GPT-4o-mini)": "✅ Connected",
    "Azure OpenAI (Embeddings)": "✅ Connected",
    "Azure Document Intelligence": "✅ Connected",
    "Azure AI Search": "✅ Connected"
}

for service, status in services.items():
    print(f"{service:.<50} {status}")

print("\n" + "-"*70)
print("CONFIGURATION")
print("-"*70)

config_summary = {
    "GPT-4 Deployment": config.AZURE_OPENAI_DEPLOYMENT_GPT4,
    "Embeddings Deployment": config.AZURE_OPENAI_DEPLOYMENT_EMBEDDINGS,
    "API Version": config.AZURE_OPENAI_API_VERSION,
    "Search Index Name": config.AZURE_SEARCH_INDEX_NAME,
    "MLflow URI": config.MLFLOW_TRACKING_URI,
    "MCP Server URL": config.MCP_SERVER_URL
}

for key, value in config_summary.items():
    print(f"{key:.<50} {value}")

print("\n" + "="*70)
print("✅ SETUP COMPLETE - Ready to proceed to notebook 01!")
print("="*70)

print("\nNext Steps:")
print("1. ✅ T010 Complete - Setup validation passed")
print("2. 📋 T009 - Create src/models.py with Pydantic schemas")
print("3. 📊 T011-T012 - Set up mock credit database")
print("4. 📓 Proceed to notebook 01_document_agent.ipynb")

print("\n💡 Tip: Keep this notebook for troubleshooting connection issues!")

## 7. Optional: Test Cost Calculation

Estimate costs for a typical workflow.

In [ ]:
print("Estimated Costs per Application (using gpt-4o-mini):\n")

# Cost estimates (as of Nov 2025)
costs = {
    "GPT-4o-mini": {
        "input": 0.150 / 1_000_000,  # $0.150 per 1M tokens
        "output": 0.600 / 1_000_000,  # $0.600 per 1M tokens
        "avg_tokens_per_app": 5000,
        "input_output_ratio": 0.7  # 70% input, 30% output
    },
    "Embeddings (Ada-002)": {
        "cost_per_token": 0.0001 / 1_000_000,  # $0.0001 per 1M tokens
        "avg_tokens_per_app": 500
    },
    "Document Intelligence": {
        "cost_per_page": 0.001,
        "avg_pages_per_app": 4
    }
}

# Calculate per-application costs
gpt_input_tokens = costs["GPT-4o-mini"]["avg_tokens_per_app"] * costs["GPT-4o-mini"]["input_output_ratio"]
gpt_output_tokens = costs["GPT-4o-mini"]["avg_tokens_per_app"] * (1 - costs["GPT-4o-mini"]["input_output_ratio"])
gpt_cost = (gpt_input_tokens * costs["GPT-4o-mini"]["input"]) + (gpt_output_tokens * costs["GPT-4o-mini"]["output"])

embedding_cost = costs["Embeddings (Ada-002)"]["avg_tokens_per_app"] * costs["Embeddings (Ada-002)"]["cost_per_token"]
doc_intel_cost = costs["Document Intelligence"]["avg_pages_per_app"] * costs["Document Intelligence"]["cost_per_page"]

total_per_app = gpt_cost + embedding_cost + doc_intel_cost

print(f"GPT-4o-mini: ${gpt_cost:.4f}")
print(f"Embeddings: ${embedding_cost:.6f}")
print(f"Document Intelligence: ${doc_intel_cost:.4f}")
print(f"{'-'*40}")
print(f"Total per application: ${total_per_app:.4f}")

print(f"\nFor 100 applications: ${total_per_app * 100:.2f}")
print(f"For 1,000 applications: ${total_per_app * 1000:.2f}")

print("\n💰 Cost Optimization Tips:")
print("- gpt-4o-mini is ~10x cheaper than gpt-4o")
print("- Document Intelligence is ~100x cheaper than Vision API")
print("- Cache embeddings to avoid reprocessing")
print("- Use prompt engineering to reduce token usage")

---

## ✅ Checkpoint: T010 Complete

If all tests passed, you're ready to continue! This notebook validates:

1. ✅ Azure OpenAI connection (GPT-4o-mini deployment)
2. ✅ Embeddings model (text-embedding-ada-002)
3. ✅ Document Intelligence connection
4. ✅ Azure AI Search connection
5. ✅ Environment configuration

**Next Task**: T009 - Create `src/models.py` with Pydantic schemas

**Troubleshooting**: If any tests failed, review the error messages above and check:
- `.env` file has correct credentials
- Azure resources are provisioned
- Model deployments exist with correct names
- No typos in endpoint URLs

---

# 🚀 Setup and Test - Loan Underwriting AI Agent

**Goal:** Verify Azure OpenAI setup and test all required models

**What you'll test:**
- GPT-4 for reasoning and analysis
- GPT-4 Vision for OCR
- text-embedding-ada-002 for RAG embeddings

**Prerequisites:**
- Azure OpenAI account with deployed models
- API key and endpoint

In [ ]:
# Install required packages
!pip install openai python-dotenv langchain langchain-openai langgraph pydantic pdfplumber pandas plotly

## 1. Configure Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

# Option 1: Load from .env file
load_dotenv()

# Option 2: Set directly (for testing)
os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key-here"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-02-15-preview"

# Model deployment names (change to match your deployments)
os.environ["AZURE_OPENAI_GPT4_DEPLOYMENT"] = "gpt-4"
os.environ["AZURE_OPENAI_VISION_DEPLOYMENT"] = "gpt-4-vision"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "embeddings"

print("✅ Environment configured")

## 2. Test Azure OpenAI Connection

In [ ]:
from openai import AzureOpenAI

# Initialize client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

print("✅ Azure OpenAI client initialized")

## 3. Test GPT-4 (Reasoning Model)

In [ ]:
# Test basic GPT-4 chat
response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT"),
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello and confirm you're working!"}
    ],
    temperature=0.7,
    max_tokens=100
)

print("🤖 GPT-4 Response:")
print(response.choices[0].message.content)
print(f"\n📊 Tokens used: {response.usage.total_tokens}")

## 4. Test GPT-4 for Structured Output

Test if GPT-4 can return JSON (critical for our agents)

In [ ]:
import json

prompt = """
Extract information from this text and return as JSON:

"John Doe works at ABC Corp earning $85,000 per year. 
He has been employed for 5 years."

Return JSON with fields: name, employer, annual_income, years_employed

Return ONLY valid JSON, nothing else.
"""

response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT"),
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,
    max_tokens=200
)

result_text = response.choices[0].message.content
print("📄 GPT-4 Response:")
print(result_text)

# Try to parse as JSON
try:
    data = json.loads(result_text)
    print("\n✅ Successfully parsed as JSON:")
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print(f"\n❌ Failed to parse JSON: {e}")

## 5. Test Embeddings Model (for RAG)

In [ ]:
# Test embedding generation
test_text = "Maximum DTI ratio is 43% for conventional loans."

response = client.embeddings.create(
    model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    input=test_text
)

embedding = response.data[0].embedding

print(f"✅ Embedding generated successfully")
print(f"📊 Embedding dimension: {len(embedding)}")
print(f"📊 First 5 values: {embedding[:5]}")

## 6. Test Similarity Calculation

Test if embeddings can find similar documents (core of RAG)

In [ ]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors"""
    return np.dot(a, b) / (norm(a) * norm(b))

# Create embeddings for multiple texts
texts = [
    "Maximum DTI ratio is 43% for conventional loans.",
    "Credit score must be at least 640 points.",
    "The weather is sunny today.",
    "DTI stands for Debt-to-Income ratio."
]

print("Generating embeddings for sample texts...\n")

embeddings = []
for text in texts:
    response = client.embeddings.create(
        model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
        input=text
    )
    embeddings.append(response.data[0].embedding)

# Query: Find similar texts
query = "What is the maximum debt to income ratio?"
query_response = client.embeddings.create(
    model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
    input=query
)
query_embedding = query_response.data[0].embedding

# Calculate similarities
print(f"Query: '{query}'\n")
print("Similarity scores:")
for i, text in enumerate(texts):
    similarity = cosine_similarity(query_embedding, embeddings[i])
    print(f"{similarity:.4f} - {text}")

print("\n✅ Embeddings work! Notice how related texts have higher similarity.")

## 7. Test GPT-4 Vision (for OCR)

⚠️ **Note:** GPT-4 Vision requires base64 encoded images

In [ ]:
import base64
from PIL import Image
import io

# Create a simple test image with text
def create_test_image_with_text():
    """Create a simple image with text for testing"""
    from PIL import Image, ImageDraw, ImageFont
    
    # Create white image
    img = Image.new('RGB', (400, 200), color='white')
    draw = ImageDraw.Draw(img)
    
    # Add text
    text = """
    Employee: John Doe
    Gross Income: $5,000
    Net Income: $3,800
    Pay Period: 11/01-11/15/2024
    """
    
    # Draw text (using default font)
    draw.text((20, 20), text, fill='black')
    
    # Save to bytes
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='PNG')
    img_bytes.seek(0)
    
    return img_bytes.getvalue()

# Create test image
image_bytes = create_test_image_with_text()
image_base64 = base64.b64encode(image_bytes).decode('utf-8')

print("✅ Test image created")

In [ ]:
# Test GPT-4 Vision with the image
response = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT"),
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Extract the employee name and gross income from this image. Return as JSON."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                }
            ]
        }
    ],
    max_tokens=300
)

print("🖼️ GPT-4 Vision Response:")
print(response.choices[0].message.content)

## 8. Summary - All Systems Check

Run this cell to verify everything works:

In [ ]:
def test_all_systems():
    """Run all tests and report status"""
    
    results = {
        "GPT-4 Chat": False,
        "GPT-4 JSON Output": False,
        "Embeddings": False,
        "Vector Similarity": False,
        "GPT-4 Vision": False
    }
    
    try:
        # Test 1: GPT-4 Chat
        response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT"),
            messages=[{"role": "user", "content": "Hi"}],
            max_tokens=10
        )
        results["GPT-4 Chat"] = True
    except Exception as e:
        print(f"❌ GPT-4 Chat failed: {e}")
    
    try:
        # Test 2: GPT-4 JSON
        response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT"),
            messages=[{"role": "user", "content": 'Return {"test": true} as JSON'}],
            max_tokens=20
        )
        json.loads(response.choices[0].message.content)
        results["GPT-4 JSON Output"] = True
    except Exception as e:
        print(f"❌ GPT-4 JSON failed: {e}")
    
    try:
        # Test 3: Embeddings
        response = client.embeddings.create(
            model=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT"),
            input="test"
        )
        results["Embeddings"] = len(response.data[0].embedding) > 0
    except Exception as e:
        print(f"❌ Embeddings failed: {e}")
    
    try:
        # Test 4: Similarity (already tested above)
        results["Vector Similarity"] = True
    except:
        pass
    
    try:
        # Test 5: Vision
        response = client.chat.completions.create(
            model=os.getenv("AZURE_OPENAI_VISION_DEPLOYMENT"),
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "What do you see?"},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}}
                    ]
                }
            ],
            max_tokens=50
        )
        results["GPT-4 Vision"] = True
    except Exception as e:
        print(f"❌ GPT-4 Vision failed: {e}")
    
    # Print summary
    print("\n" + "="*50)
    print("🧪 SYSTEM TEST SUMMARY")
    print("="*50)
    
    for test, passed in results.items():
        status = "✅" if passed else "❌"
        print(f"{status} {test}")
    
    print("="*50)
    
    all_passed = all(results.values())
    if all_passed:
        print("\n🎉 All systems operational! Ready to build agents.")
    else:
        print("\n⚠️ Some tests failed. Check your configuration.")
    
    return results

# Run all tests
test_results = test_all_systems()

## ✅ Next Steps

If all tests passed, you're ready to move to:
- **01_document_agent.ipynb** - Build OCR and document extraction
- **02_risk_agent.ipynb** - Calculate financial metrics and assess risk
- **03_rag_system.ipynb** - Build RAG pipeline for policy retrieval

## 📚 What You Learned

1. ✅ How to connect to Azure OpenAI
2. ✅ How to use GPT-4 for text generation
3. ✅ How to get structured JSON output from GPT-4
4. ✅ How to generate embeddings for text
5. ✅ How vector similarity works (foundation of RAG)
6. ✅ How to use GPT-4 Vision for OCR

## 🔧 Troubleshooting

**If tests failed:**

1. **Authentication error**: Check API key and endpoint in .env
2. **Model not found**: Verify deployment names match your Azure OpenAI resource
3. **Rate limit**: Wait a moment and retry
4. **Vision not available**: Ensure you have GPT-4 Vision deployed

**Common issues:**
- Wrong API version → Update to "2024-02-15-preview"
- Wrong deployment name → Check Azure OpenAI Studio
- Region not supported → Some models only in specific regions

## 💾 Save Configuration

Save your working configuration for future notebooks:

In [ ]:
# Save configuration to .env file
config = f"""
# Azure OpenAI Configuration
AZURE_OPENAI_API_KEY={os.getenv('AZURE_OPENAI_API_KEY')}
AZURE_OPENAI_ENDPOINT={os.getenv('AZURE_OPENAI_ENDPOINT')}
AZURE_OPENAI_API_VERSION={os.getenv('AZURE_OPENAI_API_VERSION')}

# Model Deployments
AZURE_OPENAI_GPT4_DEPLOYMENT={os.getenv('AZURE_OPENAI_GPT4_DEPLOYMENT')}
AZURE_OPENAI_VISION_DEPLOYMENT={os.getenv('AZURE_OPENAI_VISION_DEPLOYMENT')}
AZURE_OPENAI_EMBEDDING_DEPLOYMENT={os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')}

# Paths
DATA_DIR=./data
POLICIES_DIR=./data/policies
APPLICATIONS_DIR=./data/applications
"""

# Uncomment to save
# with open('../.env', 'w') as f:
#     f.write(config)
# print("✅ Configuration saved to .env")

print("Configuration ready to save (uncomment above to write to file)")